<br>

<h1 style="text-align:center;">Machine Translation (MT)</h1>
<h3 style="text-align:center;">English-to-Polish</h3>

<br>

# Initial Setup

---

In [1]:
import os
import tempfile
import shutil
from typing import Dict, List, Optional, Tuple
import torch
from datasets import load_from_disk, Dataset, load_dataset
import transformers
from transformers import *
from transformers.trainer_callback import TrainerCallback
import numpy as np
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig
import hashlib
from itertools import islice
import evaluate
from tqdm import tqdm
import pandas as pd
import nltk

c:\Users\Soheil\.conda\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Soheil\.conda\envs\llm\lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: '__init_subclass__' (from 'transformers.agents.tools') is deprecated and will be removed from version '4.51.0'. Switch to smolagents instead, with the same functionalities and similar API (https://huggingface.co/docs/smolagents/index)
  warnings.warn(warning_message, FutureWarning)


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
import warnings
warnings.filterwarnings("ignore")

<br>

# Load Dataset

---

In [4]:
# Parameters
MAX_LENGTH = 100
MAX_TRAIN_SAMPLES = 1_000_000  # Possible values: None (for all), or an integer to limit training examples
MAX_TEST_SAMPLES = 10_000    # Possible values: None (for all), or an integer to limit test examples
BATCH_SIZE = 32
MAX_STEPS = int(MAX_TRAIN_SAMPLES / BATCH_SIZE) if MAX_TRAIN_SAMPLES else 1000  # Default to 1000 if not set

In [5]:
# Load the dataset
dataset = load_dataset("open_subtitles", lang1="en", lang2="pl", streaming=True)['train']

In [6]:
# Split the dataset into train, validation and test (80% train, 10% validation, 10% test)

# Function to split the dataset into train, validation and test
def get_split_with_filter(ex):
    """Determine the split (train, validation, test) based on a hash of the example."""
    en, pl = ex["translation"]["en"], ex["translation"]["pl"]
    if len(en) < MAX_LENGTH and len(pl) < MAX_LENGTH:
        text = en + pl
        md5_hash = hashlib.md5(text.encode()).hexdigest()
        hash_val = int(md5_hash, 16) % 100
        if hash_val < 80:
            return 'train'
        elif hash_val < 90:
            return 'validation'
        else:
            return 'test'
    return None

# Create splits based on the filter function
splits = {
    split: dataset.filter(lambda ex: get_split_with_filter(ex) == split)
    for split in ['train', 'validation', 'test']
}

# Limit training examples if MAX_TRAIN_SAMPLES is specified
if MAX_TRAIN_SAMPLES is not None:
    splits['train'] = splits['train'].take(MAX_TRAIN_SAMPLES)

# Limit test examples if MAX_TEST_SAMPLES is specified
if MAX_TEST_SAMPLES is not None:
    splits['test'] = splits['test'].take(MAX_TEST_SAMPLES)

# Shuffle the training split for randomness
splits['train'] = splits['train'].shuffle(seed=42)

# Print the first 6 examples from the training split
for ex in islice(splits['train'], 6):
    print(f"English: {ex['translation']['en']}")
    print(f"Polish: {ex['translation']['pl']}")
    print("-" * 30)

English: Then I'm throwing it away.
Polish: Zatem, wywalę to do śmieci.
------------------------------
English: I'm a little surprised... the news of it was so sudden.
Polish: Jestem trochę zaskoczona... to stało się tak nagle.
------------------------------
English: Please spend the night in this room.
Polish: Otwórz drzwi. spędź proszę z nim noc.
------------------------------
English: This will all sort out, won't it?
Polish: Okaże się, prawda?
------------------------------
English: We'd... ..bonded.
Polish: Byliśmy... połączeni.
------------------------------
English: Honey. What Bang Shil did was amazing.
Polish: Bang Shil naprawdę zrobiła coś wielkiego!
------------------------------


In [7]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-3.3B", src_lang="eng_Latn", tgt_lang="pol_Latn")

# Define the tokenization function
def tokenize_function(example):
    source = example['translation']['en']
    target = example['translation']['pl']
    return tokenizer(source, text_target=target, truncation=True, max_length=MAX_LENGTH, padding=False)

# Tokenize each split
tokenized_splits = {}
for split in ['train', 'validation', 'test']:
    tokenized_splits[split] = splits[split].map(tokenize_function)

<br>

## Model Training

---

In [8]:
# Load the model
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-3.3B")

Loading checkpoint shards: 100%|██████████| 3/3 [00:10<00:00,  3.36s/it]


In [9]:
# Define the data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [10]:
# Set up training arguments
training_args = TrainingArguments(
    output_dir="./results",
    # evaluation_strategy="steps",  # Evaluate every eval_steps
    # eval_steps=200,              # Evaluate every 200 steps
    # save_strategy="steps",       # Save every save_steps
    # save_steps=200,              # Save every 200 steps
    learning_rate=5e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    max_steps=MAX_STEPS,         # Total training steps
    warmup_steps=100,            # Warm-up over 100 steps
    weight_decay=0.01,
    fp16=True,                   # Mixed precision
    gradient_checkpointing=True, # Memory optimization
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2,
    optim="adamw_torch",
)

In [11]:
# Load the BLEU metric
metric = evaluate.load("sacrebleu")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = tokenizer.pad_token_id
    label_str = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)
    return metric.compute(predictions=pred_str, references=[[label] for label in label_str])

In [12]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_splits["train"],
    eval_dataset=tokenized_splits["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    # compute_metrics=compute_metrics,
)

In [13]:
# Train the model
trainer.train()

In [19]:
# Save the final model
trainer.save_model("./final_model")

<br>

# Prediction

---

In [20]:
# Load the model weights 
MODEL_PATH = "./final_model" 

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, src_lang="eng_Latn", tgt_lang="pol_Latn")

# Load the model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

loading file sentencepiece.bpe.model
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
loading configuration file ./final_model\config.json
Model config M2M100Config {
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "architectures": [
    "M2M100ForConditionalGeneration"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 0,
  "d_model": 2048,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 8192,
  "decoder_layerdrop": 0,
  "decoder_layers": 24,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 8192,
  "encoder_layerdrop": 0,
  "encoder_layers": 24,
  "eos_token_id": 2,
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "max_length": null,
  "max_position_embeddings": 1024,
  "model_type": "m2m_100",
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "scale_embedding": true,
  "torch_dtype": "float32"

In [21]:
# Function for translating a batch of sentence  
def translate(model, tokenizer, inputs, src_lang, tgt_lang):
    MAX_LENGTH = 1024
    NUM_BEAMS = 50
    TEMPERATURE = 0.3
    
    def translate_sentences(sentences):
        tokenizer.src_lang, tokenizer.tgt_lang = src_lang, tgt_lang
        inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        outputs = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
                                 max_length=MAX_LENGTH, num_beams=NUM_BEAMS, temperature=TEMPERATURE, early_stopping=True)
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
    return [" ".join(translate_sentences(nltk.sent_tokenize(row))) for row in inputs]

In [22]:
# Test data
en_inputs = ["This is a test sentence.", "How are you doing today?"]

# Translate the test data
translations = translate(model, tokenizer, en_inputs, "eng_Latn", "pol_Latn")

translations


c:\Users\Soheil\.conda\envs\llm\lib\site-packages\transformers\generation\configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


['To jest zdanie testowe.', 'Jak się dziś macie?']

In [ ]:
### Trnalsate the test data

# Load the excel file
df = pd.read_excel("./../translations_scored.xlsx").dropna()

# List of English sentences (rows)
inputs = list(df["Sentence"])

# Initialize the translations list
translations = []

# Process each row
for row in tqdm(inputs, desc="Translating Rows", ncols=120, colour="green"):
    
    # Split the row into sentences
    sentences = nltk.sent_tokenize(row)
    
    # Translate each sentence
    translated_sentences = translate(model, tokenizer, sentences, "eng_Latn", "pol_Latn")
    
    # Combine the translated sentences back into a single row
    translated_row = " ".join(translated_sentences)
    translations.append(translated_row)

    # Print the original and translated sentences
    # print(f"Original: {row}")
    # print(f"Translated: {translated_row}")
    # print("-" * 100)

# Add the translations to the dataframe
df["Translation"] = translations

# Save the dataframe to an excel file
df.to_excel("./../translations_scored.xlsx", index=False)